# Data Engineer Case Study: Item Views Data Lake

This notebook builds a small three-layer data lake on **Databricks (Unity Catalog, Delta Lake)** from two source files,
`item.csv` and `event.csv`, and ends with the `top_item` datamart.

| Layer | Schema | Purpose |
|---|---|---|
| 1 | `raw` | One table per source file. All columns `STRING`, original column names, no transformations. |
| 2 | `curated` | Same datasets with snake_case names, proper data types, the JSON payload flattened, the parameters pivoted, and the fact table partitioned. |
| 3 | `mart` | `top_item`: views, rank and most-used platform per item and year. |

**Flow**

```
S3 (public)  ->  UC Volume (landing)  ->  raw.item / raw.event  ->  curated.item / curated.event  ->  mart.top_item
```

**How to run:** attach to serverless compute (Databricks Free Edition) and *Run all*. Every step is idempotent, so the
notebook can be re-run end to end and produces the same result.

**Design choices**
- **Spark SQL first.** All DDL, transformations, data-quality checks and validations are Spark SQL statements run
  through `spark.sql()`. Python is used only for configuration, the file download and reading the CSV files.
- Tables are created with explicit DDL (types and column comments) before loading, so the schema is the contract,
  not whatever Spark infers.
- Each load is `INSERT OVERWRITE`, a full reload, which is correct for a one-off batch. Incremental loading is
  discussed in `open_questions.md`.

## 0. Configuration

All names in one place. `workspace` is the default catalog in Databricks Free Edition.

In [0]:
CATALOG = "workspace"
RAW, CURATED, MART = "raw", "curated", "mart"

# Landing zone for the downloaded files: a Unity Catalog volume inside the raw schema
LANDING_VOLUME = "landing"
LANDING_PATH = f"/Volumes/{CATALOG}/{RAW}/{LANDING_VOLUME}"

SOURCE_BASE_URL = "https://merkle-de-interview-case-study.s3.eu-central-1.amazonaws.com/de"
SOURCE_FILES = ["item.csv", "event.csv"]

# Fully qualified schema names used in the SQL below
RAW_DB, CURATED_DB, MART_DB = f"{CATALOG}.{RAW}", f"{CATALOG}.{CURATED}", f"{CATALOG}.{MART}"

26/10/05 18:55:30 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 1. Setup: schemas and landing volume

In [0]:
for schema in (RAW_DB, CURATED_DB, MART_DB):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {RAW_DB}.{LANDING_VOLUME}")

DataFrame[]

## 2. Load the source files from S3

The bucket is public and accessed over HTTPS, so no AWS credentials are needed. Serverless compute does not allow
custom S3 (`s3a`) configuration, so the files are downloaded with the Python standard library into the landing
volume and read by Spark from there.

*Fallback:* if outbound HTTP is blocked in the workspace, upload both files through **Catalog → workspace → raw →
landing → Upload to this volume** and skip this cell.

In [0]:
import urllib.request

for file_name in SOURCE_FILES:
    target = f"{LANDING_PATH}/{file_name}"
    urllib.request.urlretrieve(f"{SOURCE_BASE_URL}/{file_name}", target)
    print(f"downloaded {file_name} -> {target}")

downloaded item.csv -> /Volumes/workspace/raw/landing/item.csv
downloaded event.csv -> /Volumes/workspace/raw/landing/event.csv


## 3. Understand the data

Both files are registered as temporary views (`item_csv`, `event_csv`) with **every column read as `STRING`**
(no type inference), so they can be explored and loaded with plain SQL.

Findings (the profiling queries below print the numbers behind them):

**`item.csv`** is the item (product) dimension, one row per item.
- `id` is stored as a float string (`2512.0`) but is a whole number, so it becomes `BIGINT`.
- `adjective` and `modifier` are optional; empty strings mean "no value" and become `NULL`.
- `name` is derived from `adjective + category + modifier`, so it is not unique; `id` is the key.
- `price` has float artefacts (`0.7800000000000001`), so it becomes `DECIMAL(10,2)`.

**`event.csv`** is the user event log (fact).
- `user_id` is also a float string, so it becomes `BIGINT`.
- `event.payload` is a JSON object with 4 keys: `event_name`, `platform`, `parameter_name`, `parameter_value`.
- The payload is a **key/value parameter collection spread over rows**: an event with N parameters appears N
  times with the same `event_id`, `event_time`, `user_id`, `event_name` and `platform`.
- 853,640 rows describe 428,364 events. Each event type has a fixed parameter set:

  | `event_name` | Parameters | Events |
  |---|---|---|
  | `view_item` | `item_id`, `referrer` | 262,786 |
  | `test_assignment` | `test_id`, `test_assignment` | 162,490 |
  | `view_user_profile` | `viewed_user_id` | 3,088 |

- `item_id` (in `view_item`) is the link to the item dimension.
- `server` appears only on `test_assignment` (backend A/B-test assignment); item views come from client platforms.
- There are no exact duplicate rows and no empty payloads.
- Events span 2013-03 to 2018-06; items were created 2013-02 to 2014-02.

In [0]:
def register_csv_view(file_name, view_name):
    # Expose a CSV file as a temporary view with every column as STRING (no type inference).
    # escape='"' handles the doubled quotes ("") inside the JSON payload column.
    (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("quote", '"')
        .option("escape", '"')
        .csv(f"{LANDING_PATH}/{file_name}")
        .createOrReplaceTempView(view_name)
    )


register_csv_view("item.csv", "item_csv")
register_csv_view("event.csv", "event_csv")

display(spark.sql("DESCRIBE event_csv"))
display(spark.sql("SELECT * FROM event_csv LIMIT 5"))

col_name,data_type,comment
event_id,string,null
event_time,string,null
user_id,string,null
event.payload,string,null


event_id,event_time,user_id,event.payload
b9de71c5c3cc4cd7a97e50b832106e5a,2017-06-26 11:23:39,178481.0,"{""event_name"":""view_item"",""platform"":""android"",""parameter_name"":""item_id"",""parameter_value"":""3526""}"
23267713c9ea44419331731f50b6a8db,2017-06-27 10:46:39,178481.0,"{""event_name"":""view_item"",""platform"":""android"",""parameter_name"":""item_id"",""parameter_value"":""1514""}"
1b7822fa7b854e01970218ae8f721fe0,2017-06-27 11:15:39,178481.0,"{""event_name"":""view_item"",""platform"":""android"",""parameter_name"":""item_id"",""parameter_value"":""3712""}"
2a7a188a626841ac94befcc419f06af4,2016-10-05 20:43:10,154133.0,"{""event_name"":""view_item"",""platform"":""android"",""parameter_name"":""item_id"",""parameter_value"":""3586""}"
631d657264cc4616a4528f759509b25d,2016-10-04 03:29:10,154133.0,"{""event_name"":""view_item"",""platform"":""android"",""parameter_name"":""item_id"",""parameter_value"":""1061""}"


In [0]:
print("Event name x parameter name:")
display(spark.sql('''
    SELECT get_json_object(`event.payload`, '$.event_name')     AS event_name,
           get_json_object(`event.payload`, '$.parameter_name') AS parameter_name,
           COUNT(*)                                             AS row_count
    FROM event_csv
    GROUP BY 1, 2
    ORDER BY row_count DESC
'''))

print("Platform per event name:")
display(spark.sql('''
    SELECT get_json_object(`event.payload`, '$.event_name') AS event_name,
           get_json_object(`event.payload`, '$.platform')   AS platform,
           COUNT(*)                                         AS row_count
    FROM event_csv
    GROUP BY 1, 2
    ORDER BY 1, row_count DESC
'''))

print("Event keys, duplicates and time range:")
display(spark.sql('''
    SELECT COUNT(*)                                       AS row_count,
           COUNT(DISTINCT event_id)                       AS distinct_event_id,
           COUNT(DISTINCT event_id, `event.payload`)      AS distinct_event_id_payload,
           COUNT(*) - COUNT(`event.payload`)              AS null_payload,
           MIN(event_time)                                AS min_event_time,
           MAX(event_time)                                AS max_event_time
    FROM event_csv
'''))

print("Item keys and creation range:")
display(spark.sql('''
    SELECT COUNT(*)            AS row_count,
           COUNT(DISTINCT id)  AS distinct_id,
           MIN(created_at)     AS min_created_at,
           MAX(created_at)     AS max_created_at
    FROM item_csv
'''))

Event name x parameter name:


event_name,parameter_name,row_count
view_item,referrer,262786
view_item,item_id,262786
test_assignment,test_id,162490
test_assignment,test_assignment,162490
view_user_profile,viewed_user_id,3088


Platform per event name:


event_name,platform,row_count
test_assignment,server,324980
view_item,web,265882
view_item,android,86998
view_item,mobile web,86728
view_item,iOS,85964
view_user_profile,web,1539
view_user_profile,iOS,550
view_user_profile,android,503
view_user_profile,mobile web,496


Event keys, duplicates and time range:


row_count,distinct_event_id,distinct_event_id_payload,null_payload,min_event_time,max_event_time
853640,428364,853640,0,2013-03-09 09:28:22,2018-06-01 23:57:54


Item keys and creation range:


row_count,distinct_id,min_created_at,max_created_at
2198,2198,2013-02-22 14:29:32,2014-02-25 23:57:08


## 4. Layer 1: `raw`

- One table per source file.
- Every column is `STRING`; column names are kept exactly as in the file (including `event.payload`).
- No transformations: no trimming, no casting, no deduplication. This layer is a faithful, re-playable copy of
  the source.

The source column `event.payload` contains a `.`, which Spark would otherwise read as struct-field access, so
it is always quoted with backticks. `raw.event` also enables Delta *column mapping*
(`delta.columnMapping.mode = 'name'`), which guarantees that source column names with special characters are
stored exactly as delivered.

In [0]:
spark.sql(f'''
    CREATE OR REPLACE TABLE {RAW_DB}.item (
        adjective   STRING,
        category    STRING,
        created_at  STRING,
        id          STRING,
        modifier    STRING,
        name        STRING,
        price       STRING
    )
    USING DELTA
    COMMENT 'Layer 1: item.csv as delivered, all columns STRING'
''')

spark.sql(f'''
    CREATE OR REPLACE TABLE {RAW_DB}.event (
        event_id         STRING,
        event_time       STRING,
        user_id          STRING,
        `event.payload`  STRING
    )
    USING DELTA
    COMMENT 'Layer 1: event.csv as delivered, all columns STRING'
    TBLPROPERTIES (
        'delta.columnMapping.mode' = 'name',
        'delta.minReaderVersion'   = '2',
        'delta.minWriterVersion'   = '5'
    )
''')

DataFrame[]

In [0]:
spark.sql(f'''
    INSERT OVERWRITE TABLE {RAW_DB}.item
    SELECT adjective, category, created_at, id, modifier, name, price
    FROM item_csv
''')

spark.sql(f'''
    INSERT OVERWRITE TABLE {RAW_DB}.event
    SELECT event_id, event_time, user_id, `event.payload`
    FROM event_csv
''')

display(spark.sql(f'''
    SELECT 'raw.item'  AS table_name, COUNT(*) AS row_count FROM {RAW_DB}.item
    UNION ALL
    SELECT 'raw.event' AS table_name, COUNT(*) AS row_count FROM {RAW_DB}.event
'''))

table_name,row_count
raw.item,2198
raw.event,853640


## 5. Layer 2: `curated`

**Naming convention:** `snake_case`, singular table names, `<entity>_id` for keys, `_ts` / `_at` for timestamps,
`_date` / `_year` for date parts.

### 5.1 `curated.item` (dimension)

| Raw column | Curated column | Type | Rule |
|---|---|---|---|
| `id` | `item_id` | `BIGINT` | `'2512.0'` → `2512` |
| `name` | `item_name` | `STRING` | trimmed |
| `category` | `category` | `STRING` | trimmed |
| `adjective` | `adjective` | `STRING` | empty → `NULL` |
| `modifier` | `modifier` | `STRING` | empty → `NULL` |
| `price` | `price` | `DECIMAL(10,2)` | rounds float artefacts |
| `created_at` | `created_at` | `TIMESTAMP` | `yyyy-MM-dd HH:mm:ss` |

`try_cast` returns `NULL` instead of failing on a malformed value; such rows are caught by the checks below.
The table is small (about 2.2k rows), so it is **not partitioned**; partitioning would only create tiny files.

In [0]:
spark.sql(f'''
    CREATE OR REPLACE TABLE {CURATED_DB}.item (
        item_id     BIGINT         NOT NULL COMMENT 'Item identifier (business key)',
        item_name   STRING                  COMMENT 'Display name: adjective + category + modifier',
        category    STRING                  COMMENT 'Item category, e.g. widget, gadget',
        adjective   STRING                  COMMENT 'Optional descriptive adjective',
        modifier    STRING                  COMMENT 'Optional modifier, e.g. charger, refill',
        price       DECIMAL(10,2)           COMMENT 'Item price',
        created_at  TIMESTAMP               COMMENT 'Item creation timestamp'
    )
    USING DELTA
    COMMENT 'Layer 2: item dimension, typed and cleaned. Grain: one row per item_id'
''')

DataFrame[]

In [0]:
spark.sql(f'''
    INSERT OVERWRITE TABLE {CURATED_DB}.item
    SELECT
        try_cast(try_cast(id AS DOUBLE) AS BIGINT)            AS item_id,
        nullif(trim(name), '')                                AS item_name,
        nullif(trim(category), '')                            AS category,
        nullif(trim(adjective), '')                           AS adjective,
        nullif(trim(modifier), '')                            AS modifier,
        try_cast(price AS DECIMAL(10,2))                      AS price,
        to_timestamp(created_at, 'yyyy-MM-dd HH:mm:ss')       AS created_at
    FROM {RAW_DB}.item
    WHERE try_cast(try_cast(id AS DOUBLE) AS BIGINT) IS NOT NULL
''')

display(spark.sql(f"SELECT * FROM {CURATED_DB}.item LIMIT 5"))

item_id,item_name,category,adjective,modifier,price,created_at
2512,fuzzy contraption carrying_case,contraption,fuzzy,carrying_case,150.00,2014-01-15T21:36:09.000Z
482,instrument refill,instrument,null,refill,35.20,2013-05-14T05:20:50.000Z
2446,industrial-strength module,module,industrial-strength,null,300.00,2014-02-04T19:28:32.000Z
1312,digital tool carrying_case,tool,digital,carrying_case,16.50,2013-02-25T12:23:18.000Z
3556,miniature device cleaner,device,miniature,cleaner,16.50,2013-08-05T17:20:45.000Z


### 5.2 `curated.event` (fact)

Two steps turn the raw log into a clean fact table, both in one SQL statement:

1. **Flatten** (CTE `parsed`): `event.payload` is parsed with `from_json` and an **explicit schema** into
   top-level columns.
2. **Transform / pivot** (final `SELECT`): the key/value parameter rows of one event are pivoted with
   `MAX(CASE WHEN parameter_name = … THEN parameter_value END)` into **one typed column per parameter**, so the
   grain becomes **one row per event**. Downstream queries use `item_id` directly instead of filtering
   `parameter_name = 'item_id'`, and counting rows counts events.

| Raw column | Curated column | Type | Rule |
|---|---|---|---|
| `event_id` | `event_id` | `STRING` | as is (hex UUID) |
| `event_time` | `event_ts` | `TIMESTAMP` | `yyyy-MM-dd HH:mm:ss` |
| `user_id` | `user_id` | `BIGINT` | `'178481.0'` → `178481` |
| `event.payload.event_name` | `event_name` | `STRING` | flattened |
| `event.payload.platform` | `platform` | `STRING` | flattened, lower-cased (`iOS` → `ios`) |
| `parameter_value` where `parameter_name = 'item_id'` | `item_id` | `BIGINT` | pivoted (`view_item`) |
| `parameter_value` where `parameter_name = 'referrer'` | `referrer` | `STRING` | pivoted (`view_item`) |
| `parameter_value` where `parameter_name = 'test_id'` | `test_id` | `INT` | pivoted (`test_assignment`) |
| `parameter_value` where `parameter_name = 'test_assignment'` | `test_assignment` | `INT` | pivoted (`test_assignment`), test group |
| `parameter_value` where `parameter_name = 'viewed_user_id'` | `viewed_user_id` | `BIGINT` | pivoted (`view_user_profile`) |
| *(derived)* | `event_date` | `DATE` | from `event_ts` |
| *(derived)* | `event_year` | `INT` | from `event_ts`, **partition column** |

The event attributes (`event_ts`, `user_id`, `event_name`, `platform`) are identical on all parameter rows of an
event, so they are part of the `GROUP BY`. If they ever differed, `event_id` would no longer be unique and
validation check 3 would fail. An unknown parameter name in new data is reported by the quality check and
also fails validation, instead of being silently dropped.

**Partitioning:** `PARTITIONED BY (event_year)`.
- Year is the grain of the datamart, so mart queries (and typical "last year" analyses) prune whole partitions.
- Low cardinality (6 years, 35k–113k events each) gives well-sized files. Partitioning by `event_date` would
  create roughly 1,900 daily partitions of about 200 events each, the classic small-files problem for a table
  of this size.
- On a growing production table I would consider `event_date` (or liquid clustering on `event_date, item_id`)
  once daily volumes justify it.

**Data quality rules**
- Exact duplicate rows are removed (`SELECT DISTINCT`) before pivoting. There are none in the current file, but a
  re-delivered file would otherwise double-count events.
- Rows whose payload cannot be parsed, or whose timestamp is invalid, are *not* silently dropped. They are counted
  and reported below; in production they would go to a quarantine table.

In [0]:
spark.sql(f'''
    CREATE OR REPLACE TABLE {CURATED_DB}.event (
        event_id         STRING  NOT NULL COMMENT 'Event identifier (unique)',
        event_ts         TIMESTAMP        COMMENT 'Event timestamp',
        user_id          BIGINT           COMMENT 'User who triggered the event',
        event_name       STRING           COMMENT 'Event type: view_item, test_assignment, view_user_profile',
        platform         STRING           COMMENT 'Platform, lower-case, e.g. android, ios, web, server',
        item_id          BIGINT           COMMENT 'Viewed item (view_item)',
        referrer         STRING           COMMENT 'Traffic source of the item view (view_item)',
        test_id          INT              COMMENT 'A/B test identifier (test_assignment)',
        test_assignment  INT              COMMENT 'A/B test group the user was assigned to (test_assignment)',
        viewed_user_id   BIGINT           COMMENT 'Profile that was viewed (view_user_profile)',
        event_date       DATE             COMMENT 'Event date (derived from event_ts)',
        event_year       INT              COMMENT 'Event year (derived from event_ts), partition column'
    )
    USING DELTA
    PARTITIONED BY (event_year)
    COMMENT 'Layer 2: user event fact, payload flattened and parameters pivoted. Grain: one row per event_id'
''')

DataFrame[]

In [0]:
spark.sql(f'''
    WITH deduplicated AS (
        SELECT DISTINCT * FROM {RAW_DB}.event
    ),
    parsed AS (                                  -- step 1: flatten, one row per event parameter
        SELECT
            event_id,
            to_timestamp(event_time, 'yyyy-MM-dd HH:mm:ss')                 AS event_ts,
            try_cast(try_cast(user_id AS DOUBLE) AS BIGINT)                 AS user_id,
            from_json(`event.payload`,
                      'event_name STRING, platform STRING, parameter_name STRING, parameter_value STRING'
            )                                                               AS payload
        FROM deduplicated
        WHERE event_id IS NOT NULL
    )
    INSERT OVERWRITE TABLE {CURATED_DB}.event
    SELECT                                       -- step 2: pivot, one row per event
        event_id,
        event_ts,
        user_id,
        payload.event_name                                                                                AS event_name,
        lower(trim(payload.platform))                                                                     AS platform,
        try_cast(MAX(CASE WHEN payload.parameter_name = 'item_id'         THEN payload.parameter_value END) AS BIGINT) AS item_id,
        MAX(CASE WHEN payload.parameter_name = 'referrer'                 THEN payload.parameter_value END)            AS referrer,
        try_cast(MAX(CASE WHEN payload.parameter_name = 'test_id'         THEN payload.parameter_value END) AS INT)    AS test_id,
        try_cast(MAX(CASE WHEN payload.parameter_name = 'test_assignment' THEN payload.parameter_value END) AS INT)    AS test_assignment,
        try_cast(MAX(CASE WHEN payload.parameter_name = 'viewed_user_id'  THEN payload.parameter_value END) AS BIGINT) AS viewed_user_id,
        to_date(event_ts)                                                                                 AS event_date,
        year(event_ts)                                                                                    AS event_year
    FROM parsed
    GROUP BY event_id, event_ts, user_id, payload.event_name, lower(trim(payload.platform))
''')

display(spark.sql(f"SELECT * FROM {CURATED_DB}.event LIMIT 5"))

event_id,event_ts,user_id,event_name,platform,item_id,referrer,test_id,test_assignment,viewed_user_id,event_date,event_year
afc029154c67458fa30a3a0c181a9f06,2018-02-09T19:06:01.000Z,253835,view_item,android,3863,user_wishlist,null,null,null,2018-02-09,2018
12a32582528149ca9961457955581fdc,2018-01-10T04:34:36.000Z,225214,view_item,ios,396,shopping_cart,null,null,null,2018-01-10,2018
22c32dcac61a462296ed24ee42d91738,2018-03-25T10:46:06.000Z,260935,view_item,web,3518,shopping_cart,null,null,null,2018-03-25,2018
e09aceea515d4fc3a4f2a6f94a138fbe,2018-03-25T02:05:31.000Z,263651,view_item,android,2232,item_page,null,null,null,2018-03-25,2018
3f326ac74cd240e78c3b8eec31b44940,2018-02-04T12:23:01.000Z,235782,view_item,web,3155,home,null,null,null,2018-02-04,2018


In [0]:
# Data quality report for layer 2 (reported, not silently fixed)
print("Quality counters:")
display(spark.sql(f'''
    SELECT COUNT(*)                                                         AS events,
           COUNT_IF(event_name IS NULL)                                     AS unparseable_payload,
           COUNT_IF(event_ts IS NULL)                                       AS invalid_timestamp,
           COUNT_IF(user_id IS NULL)                                        AS missing_user_id,
           COUNT_IF(event_name = 'view_item' AND item_id IS NULL)           AS view_without_item_id
    FROM {CURATED_DB}.event
'''))

print("Parameters not mapped to a curated column (should be empty):")
display(spark.sql(f'''
    SELECT get_json_object(`event.payload`, '$.parameter_name') AS parameter_name,
           COUNT(*)                                             AS row_count
    FROM {RAW_DB}.event
    WHERE get_json_object(`event.payload`, '$.parameter_name')
          NOT IN ('item_id', 'referrer', 'test_id', 'test_assignment', 'viewed_user_id')
    GROUP BY 1
'''))

print("Item views referencing an item missing from curated.item:")
display(spark.sql(f'''
    SELECT COUNT(*) AS orphan_views
    FROM {CURATED_DB}.event e
    LEFT ANTI JOIN {CURATED_DB}.item i ON e.item_id = i.item_id
    WHERE e.event_name = 'view_item' AND e.item_id IS NOT NULL
'''))

print("Events per partition:")
display(spark.sql(f'''
    SELECT event_year, COUNT(*) AS events
    FROM {CURATED_DB}.event
    GROUP BY event_year
    ORDER BY event_year
'''))

Quality counters:


events,unparseable_payload,invalid_timestamp,missing_user_id,view_without_item_id
428364,0,0,0,0


Parameters not mapped to a curated column (should be empty):


parameter_name,row_count


Item views referencing an item missing from curated.item:


orphan_views
0


Events per partition:


event_year,events
2013,50920
2014,112541
2015,49759
2016,102944
2017,77616
2018,34584


## 6. Layer 3: `mart.top_item`

**Question answered:** *Which items are viewed most, per year, and on which platform?*

**Grain:** one row per `item_id` × `view_year`.

| Column | Type | Definition |
|---|---|---|
| `view_year` | `INT` | Calendar year of the view events |
| `item_id` | `BIGINT` | Item identifier |
| `item_name` | `STRING` | Item name from `curated.item` (`NULL` if the item is unknown) |
| `category` | `STRING` | Item category from `curated.item` |
| `total_views` | `BIGINT` | Number of `view_item` events for the item in that year |
| `view_rank` | `INT` | Rank of the item within the year by `total_views`, descending (1 = most viewed) |
| `top_platform` | `STRING` | Platform with the most views of the item in that year |
| `top_platform_views` | `BIGINT` | Number of views on `top_platform` |

**Business rules**
- **What is a view:** an event in `curated.event` with `event_name = 'view_item'` and a non-null `item_id`.
  The fact table has one row per event, so one row is one view.
- **Every item and year:** the mart contains all items × all years that have view events. Items without views in
  a year get `total_views = 0` and `top_platform = NULL`, so "never viewed" is visible instead of missing.
  Viewed item IDs absent from the item dimension would be kept too (with `item_name = NULL`) so totals always
  reconcile with the fact table; the current data has none.
- **Rank:** `RANK()` (standard competition ranking: 1, 2, 2, 4). Items with equal views share the rank, and the
  next rank is skipped, so the rank still says "how many items had more views".
- **Most used platform, ties:** if two platforms have the same number of views, the alphabetically first platform
  is chosen, which makes the result deterministic.

**Query structure** (one CTE per step):

| CTE | Step |
|---|---|
| `views` | item view events |
| `views_per_year` | total views per item and year |
| `platform_ranked` | views per item, year and platform, numbered by views (ties: alphabetical) |
| `item_year_grid` | every item × every year with views |
| final `SELECT` | join everything, fill zero views, `RANK()` within year |

In [0]:
spark.sql(f'''
    CREATE OR REPLACE TABLE {MART_DB}.top_item (
        view_year           INT     NOT NULL COMMENT 'Calendar year of the view events',
        item_id             BIGINT  NOT NULL COMMENT 'Item identifier',
        item_name           STRING           COMMENT 'Item name (NULL if item unknown in curated.item)',
        category            STRING           COMMENT 'Item category',
        total_views         BIGINT  NOT NULL COMMENT 'Number of view_item events for the item in the year',
        view_rank           INT     NOT NULL COMMENT 'RANK() of the item within the year by total_views desc',
        top_platform        STRING           COMMENT 'Platform with most views of the item in the year (ties: alphabetical)',
        top_platform_views  BIGINT           COMMENT 'Number of views on top_platform'
    )
    USING DELTA
    COMMENT 'Layer 3: top_item datamart. Grain: one row per item_id and view_year'
''')

DataFrame[]

In [0]:
spark.sql(f'''
    WITH views AS (
        SELECT item_id, event_year AS view_year, platform
        FROM {CURATED_DB}.event
        WHERE event_name = 'view_item' AND item_id IS NOT NULL
    ),
    views_per_year AS (
        SELECT item_id, view_year, COUNT(*) AS total_views
        FROM views
        GROUP BY item_id, view_year
    ),
    platform_ranked AS (
        SELECT item_id,
               view_year,
               platform,
               COUNT(*) AS platform_views,
               ROW_NUMBER() OVER (PARTITION BY item_id, view_year
                                  ORDER BY COUNT(*) DESC, platform ASC) AS platform_rn
        FROM views
        GROUP BY item_id, view_year, platform
    ),
    item_year_grid AS (
        SELECT i.item_id, y.view_year
        FROM (SELECT item_id FROM {CURATED_DB}.item
              UNION
              SELECT item_id FROM views) AS i
        CROSS JOIN (SELECT DISTINCT view_year FROM views) AS y
    )
    INSERT OVERWRITE TABLE {MART_DB}.top_item
    SELECT
        g.view_year,
        g.item_id,
        it.item_name,
        it.category,
        COALESCE(v.total_views, 0)                                              AS total_views,
        RANK() OVER (PARTITION BY g.view_year
                     ORDER BY COALESCE(v.total_views, 0) DESC)                  AS view_rank,
        p.platform                                                              AS top_platform,
        p.platform_views                                                        AS top_platform_views
    FROM item_year_grid g
    LEFT JOIN views_per_year  v  ON v.item_id = g.item_id AND v.view_year = g.view_year
    LEFT JOIN platform_ranked p  ON p.item_id = g.item_id AND p.view_year = g.view_year AND p.platform_rn = 1
    LEFT JOIN {CURATED_DB}.item it ON it.item_id = g.item_id
''')

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
print("Top 5 items per year:")
display(spark.sql(f'''
    SELECT *
    FROM {MART_DB}.top_item
    WHERE view_rank <= 5
    ORDER BY view_year, view_rank, item_id
'''))

Top 5 items per year:


view_year,item_id,item_name,category,total_views,view_rank,top_platform,top_platform_views
2013,2086,organic device,device,12,1,web,9
2013,3517,prize-winning widget wrapper,widget,12,1,web,4
2013,3063,fuzzy gadget wrapper,gadget,11,3,web,7
2013,3200,prize-winning widget refill,widget,10,4,web,6
2013,3309,industrial-strength dongle,dongle,10,4,mobile web,4
2013,3538,organic dongle,dongle,10,4,web,5
2013,3978,fuzzy dongle,dongle,10,4,android,3
2014,244,miniature contraption,contraption,29,1,web,11
2014,1353,mechanism,mechanism,29,1,web,11
2014,27,aerodynamic mechanism,mechanism,28,3,web,10


## 7. Validation

Each check is a SQL query that returns the number of **violations**; every count must be `0`.

| # | Check |
|---|---|
| 1 | Raw tables contain exactly the rows of the source files |
| 2 | Curated item keeps every raw item, and `item_id` is unique |
| 3 | Curated event has one row per distinct raw `event_id`, and every parameter is mapped |
| 4 | The mart has exactly one row per `item_id` × `view_year` |
| 5 | Mart total views equal the number of view events in the fact table, per year |
| 6 | Rank 1 in each year has the maximum views of that year |

In [0]:
checks = {
    "1a raw.item row count = item.csv": f'''
        SELECT ABS((SELECT COUNT(*) FROM {RAW_DB}.item) - (SELECT COUNT(*) FROM item_csv))''',
    "1b raw.event row count = event.csv": f'''
        SELECT ABS((SELECT COUNT(*) FROM {RAW_DB}.event) - (SELECT COUNT(*) FROM event_csv))''',
    "2a curated.item keeps every raw item": f'''
        SELECT ABS((SELECT COUNT(*) FROM {CURATED_DB}.item) - (SELECT COUNT(*) FROM {RAW_DB}.item))''',
    "2b curated.item item_id unique": f'''
        SELECT COUNT(*) - COUNT(DISTINCT item_id) FROM {CURATED_DB}.item''',
    "3a curated.event = distinct raw event_id": f'''
        SELECT ABS((SELECT COUNT(*) FROM {CURATED_DB}.event)
                 - (SELECT COUNT(DISTINCT event_id) FROM {RAW_DB}.event))''',
    "3b curated.event event_id unique": f'''
        SELECT COUNT(*) - COUNT(DISTINCT event_id) FROM {CURATED_DB}.event''',
    "3c every parameter mapped": f'''
        SELECT COUNT(*) FROM {RAW_DB}.event
        WHERE get_json_object(`event.payload`, '$.parameter_name')
              NOT IN ('item_id', 'referrer', 'test_id', 'test_assignment', 'viewed_user_id')''',
    "4 mart grain item_id x view_year": f'''
        SELECT COUNT(*) - COUNT(DISTINCT item_id, view_year) FROM {MART_DB}.top_item''',
    "5 mart views reconcile with fact per year": f'''
        SELECT COUNT(*)
        FROM (SELECT event_year AS view_year, COUNT(*) AS views
              FROM {CURATED_DB}.event
              WHERE event_name = 'view_item' AND item_id IS NOT NULL
              GROUP BY event_year) f
        FULL OUTER JOIN
             (SELECT view_year, SUM(total_views) AS views
              FROM {MART_DB}.top_item
              GROUP BY view_year) m
        ON f.view_year = m.view_year
        WHERE f.views IS NULL OR m.views IS NULL OR f.views <> m.views''',
    "6 rank 1 has max views": f'''
        SELECT COUNT(*)
        FROM (SELECT view_year,
                     MAX(total_views)                                    AS max_views,
                     MIN(CASE WHEN view_rank = 1 THEN total_views END)   AS rank_one_views
              FROM {MART_DB}.top_item
              GROUP BY view_year)
        WHERE max_views <> rank_one_views''',
}

failed = []
for name, query in checks.items():
    violations = spark.sql(query).first()[0]
    print(f"{'OK  ' if violations == 0 else 'FAIL'} {name}: {violations} violations")
    if violations != 0:
        failed.append(name)

assert not failed, f"Validation failed: {failed}"
print("All validation checks passed.")

OK   1a raw.item row count = item.csv: 0 violations
OK   1b raw.event row count = event.csv: 0 violations
OK   2a curated.item keeps every raw item: 0 violations
OK   2b curated.item item_id unique: 0 violations
OK   3a curated.event = distinct raw event_id: 0 violations
OK   3b curated.event event_id unique: 0 violations
OK   3c every parameter mapped: 0 violations
OK   4 mart grain item_id x view_year: 0 violations
OK   5 mart views reconcile with fact per year: 0 violations
OK   6 rank 1 has max views: 0 violations
All validation checks passed.
